In [1]:
import sys
sys.path.insert(0,"src")
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from subagents import rag_subagent
from langchain_core.messages import HumanMessage

In [3]:
r = rag_subagent["runnable"].invoke({"messages": [HumanMessage(content="what does the retrieval index module do?")]})

D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampli

In [4]:
print(r["messages"][-1].content)

Based on the provided context, the retrieval index module (built and loaded via `src/retrieval/index.py`) handles document ingestion and vector retrieval for the P2 RAG agent [src\retrieval\__init__.py, src\retrieval\index.py]. Specifically, it chunks source documents, embeds them, and persists them to disk (`build_index`), as well as loads the persisted index to return a retriever over it (`get_retriever`) [src\retrieval\index.py].


In [6]:
from deepagents import create_deep_agent
from models.llm import lite_model

In [12]:
from prompts.template.builder import build_system_prompt
prompt = build_system_prompt(
    assistant_name="Coordinator",
    role="a coordinator who delegates to specialized subagents",
    domain="this codebase and its indexed corpus",
    mission="Answer questions about the project by delegating to the right subagent.",
    tools=(
        """Use rag_subagent (via the task tool) for ANY question about content in the indexed document corpus (code, docs, notes about this project). Do not use filesystem tools (glob/ls/grep) to answer such questions — always delegate first."""
    ),
    output_format="Return the subagents answer directly.",
    constraints="Always delegate the corpus-content questions to rag_subagent first."

)

In [13]:
agent = create_deep_agent(
    model=lite_model,
    system_prompt=prompt,
    subagents=[rag_subagent]
)

In [ ]:
r2 = agent.invoke(
    {
        "messages": HumanMessage(
            content="what does the retrieval index module do?"
        )
    }
)

In [22]:
from pprint import pprint
pprint(r2["messages"])

[HumanMessage(content='what does the retrieval index module do?', additional_kwargs={}, response_metadata={}, id='c82df623-6ee8-40bb-86e3-820a1b0ca511'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'task', 'arguments': '{"subagent_type": "rag_subagent", "description": "Explain what the retrieval index module does in this project based on the indexed document corpus."}'}, '__gemini_function_call_thought_signatures__': {'call_2779439': 'El4KXAERTTIPvPZJjJW8X0EXPeucTxT0T+vpMjHgWQd9S4VxBwWj2ksupnC6TUf2IdUxsR2eTXFrHKWRyLogfudxMKHxi0+efpFg2ofEH96gyLgwMMNSvSNll29KjxiC'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a03475-3462-7012-a6ed-0df4bcd39ef9-0', tool_calls=[{'name': 'task', 'args': {'subagent_type': 'rag_subagent', 'description': 'Explain what the retrieval index module does in this project based on the indexed document corpus.'}, 'id': 'call_2779439', 

In [21]:
for m in r2["messages"]:
    if getattr(m, "tool_calls", None):
        print(m.tool_calls)

[{'name': 'task', 'args': {'subagent_type': 'rag_subagent', 'description': 'Explain what the retrieval index module does in this project based on the indexed document corpus.'}, 'id': 'call_2779439', 'type': 'tool_call'}]
